In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time
from numba import njit

# ================== 1. 物理内核：带能级记忆的演化引擎 ==================
@njit(fastmath=True, nogil=True)
def cross_scale_kernel(u_c, k_static, target_n, steps, n_bins):
    """
    u_c: 临界点 1.5436...
    k_static: 该核负责测试的 k 强度
    target_n: 该核负责的能级锚点（如 n=5000）
    """
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    
    # 仿真老化：模拟系统演化到 target_n 时的冷却状态
    # 这里的 i + target_n 是关键，让算子“感知”它身处高能级
    for i in range(steps + 1000000):
        ln_term = np.log(i + target_n + 100)
        mu_eff = u_c + k_static / (ln_term**2)
        
        x = 1 - mu_eff * x * x
        
        # 边界物理保护
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
        
        if i > 1000000:
            current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
            if 0 <= current_bin < n_bins:
                # 记录状态转移
                if i == 1000001: last_bin = current_bin
                counts[last_bin, current_bin] += 1
                last_bin = current_bin
                
    return counts

# ================== 2. 测绘 Worker ==================
def survey_worker(task):
    target_n, k_val = task
    U_C = 1.543689012
    N_BINS = 8000  # 测绘阶段 8000 维足以捕捉拓扑特征
    STEPS = 2000000 # 200万步采样，兼顾精度与速度
    
    SAVE_DIR = "riemann_10k_survey_cross_scale"
    
    try:
        t0 = time.time()
        counts = cross_scale_kernel(U_C, k_val, target_n, STEPS, N_BINS)
        
        # 构建稀疏矩阵并归一化
        P = sp.csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P.data /= row_sums[P.indices]
        
        # 提取特征值（扫雷阶段提取 100 个点即可）
        vals, _ = eigs(P, k=100, which='LM', tol=1e-4)
        phases = np.sort(np.angle(vals[np.abs(vals) > 0.4]))
        
        # 文件名：包含 n 锚点和 k 值，这是后续拟合公式的“坐标”
        filename = os.path.join(SAVE_DIR, f"n_{int(target_n)}_k_{k_val:.4f}.npy")
        np.save(filename, phases)
        
        return f"DONE | n={target_n:5} | k={k_val:.4f} | {time.time()-t0:.1f}s"
    except Exception as e:
        return f"FAIL | n={target_n:5} | k={k_val:.4f} | {str(e)}"

# ================== 3. 256 核战术调度 ==================
if __name__ == "__main__":
    SAVE_DIR = "riemann_10k_survey_cross_scale"
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)

    # --- 🌟 跨尺度测绘策略 🌟 ---
    # 我们把 10,000 个能级分成 32 个大站，每站用 8 个不同的 k 值进行探测
    # 32 * 8 = 256，刚好占满您的怪兽核心
    n_stations = np.linspace(100, 10000, 32)
    k_probes = np.linspace(4.5, 9.0, 8)
    
    tasks = []
    for n in n_stations:
        for k in k_probes:
            tasks.append((n, k))

    # 480G 内存，我们直接开启 128 并发（留一半核心处理 ARPACK 矩阵运算）
    BATCH_SIZE = 128 
    
    print(f"="*60)
    print(f"🚀 启动 256 核全标度拓扑测绘：跨维度空投")
    print(f">>> 目标能级：100 -> 10,000 | 探测范围：k={k_probes[0]}-{k_probes[-1]}")
    print(f"="*60)
    
    start_total = time.time()
    with mp.Pool(processes=BATCH_SIZE) as pool:
        results = pool.map(survey_worker, tasks)
        
    for res in results:
        print(f"  {res}")
        
    print(f"\n✅ 测绘圆满完成！总耗时: {(time.time()-start_total)/60:.2f} 分钟")
    print(f">>> 数据已存入: {SAVE_DIR}，现在可以去拟合那条 4.7 的曲线了！")

🚀 启动 256 核全标度拓扑测绘：跨维度空投
>>> 目标能级：100 -> 10,000 | 探测范围：k=4.5-9.0
  DONE | n=100.0 | k=4.5000 | 260.7s
  DONE | n=100.0 | k=5.1429 | 279.4s
  DONE | n=100.0 | k=5.7857 | 223.1s
  DONE | n=100.0 | k=6.4286 | 89.9s
  DONE | n=100.0 | k=7.0714 | 155.6s
  DONE | n=100.0 | k=7.7143 | 231.8s
  DONE | n=100.0 | k=8.3571 | 186.5s
  DONE | n=100.0 | k=9.0000 | 253.3s
  DONE | n=419.35483870967744 | k=4.5000 | 219.4s
  DONE | n=419.35483870967744 | k=5.1429 | 234.9s
  DONE | n=419.35483870967744 | k=5.7857 | 256.9s
  DONE | n=419.35483870967744 | k=6.4286 | 198.9s
  DONE | n=419.35483870967744 | k=7.0714 | 288.9s
  DONE | n=419.35483870967744 | k=7.7143 | 229.7s
  DONE | n=419.35483870967744 | k=8.3571 | 265.5s
  DONE | n=419.35483870967744 | k=9.0000 | 191.7s
  DONE | n=738.7096774193549 | k=4.5000 | 223.6s
  DONE | n=738.7096774193549 | k=5.1429 | 295.5s
  DONE | n=738.7096774193549 | k=5.7857 | 256.4s
  DONE | n=738.7096774193549 | k=6.4286 | 254.3s
  DONE | n=738.7096774193549 | k=7.0714 | 286